# G17 — external validation on BrEaST (Poland, 4 scanners, never seen in training)
Attach: **G17_loss_search_R4** output (trained `best.pt` checkpoints) + a BrEaST-Lesions USG dataset (images + `*_tumor*.png` masks). Inference only (CPU or GPU). Masks with several tumours are merged; normal scans (no tumour mask) are skipped. Same preprocessing as training: grayscale → 256×256 (INTER_AREA) → (x/255−0.5)/0.5; metrics at 256×256 exactly like the internal test.

In [ ]:
!pip -q install segmentation-models-pytorch safetensors

In [ ]:
import glob, os, re, shutil, sys, numpy as np, pandas as pd, cv2, torch
src = sorted(glob.glob('/kaggle/input/**/g17core.py', recursive=True))[0]; shutil.copy(src, 'g17core.py')
sys.path.insert(0, '.'); import g17core as g
RUNS = sorted(d for d in glob.glob('/kaggle/input/**/runs', recursive=True) if os.path.isdir(d))[0]
pngs = [p for p in glob.glob('/kaggle/input/**/*.png', recursive=True) if '/runs/' not in p]
masks = [p for p in pngs if re.search(r'_tumou?r', os.path.basename(p), re.I)]
others = [p for p in pngs if re.search(r'_(other|mask)', os.path.basename(p), re.I)]
imgs = {}
for p in pngs:
    b = os.path.basename(p)
    if re.search(r'_(tumou?r|other|mask)', b, re.I): continue
    imgs[os.path.splitext(b)[0]] = p
groups = {}
for m in masks:
    k = re.split(r'_tumou?r', os.path.splitext(os.path.basename(m))[0], flags=re.I)[0]
    groups.setdefault(k, []).append(m)
cases = sorted(k for k in groups if k in imgs)
print(len(pngs), 'png |', len(imgs), 'images |', len(groups), 'with tumour masks |', len(cases), 'matched |', len(others), 'other-masks ignored')
I, M, ids, native = [], [], [], {}
for k in cases:
    im = cv2.imread(imgs[k], cv2.IMREAD_GRAYSCALE)
    mk = np.zeros(im.shape, np.uint8)
    for m in groups[k]:
        mm = cv2.imread(m, cv2.IMREAD_GRAYSCALE)
        if mm.shape != im.shape: mm = cv2.resize(mm, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
        mk |= ((mm > 127) * 255).astype(np.uint8)
    if mk.max() == 0: continue
    native[k] = (im, mk > 0)
    I.append(cv2.resize(im, (g.IMG, g.IMG), interpolation=cv2.INTER_AREA))
    M.append(cv2.resize(mk, (g.IMG, g.IMG), interpolation=cv2.INTER_NEAREST)); ids.append(k)
ext = {"img": torch.from_numpy(np.stack(I))[:, None].to(g.DEVICE), "msk": torch.from_numpy(np.stack(M))[:, None].to(g.DEVICE),
       "dataset": ["BrEaST"] * len(ids), "image_id": ids}
nles = {k: len(groups[k]) for k in ids}
print('BrEaST scans evaluated:', len(ids), '| multi-lesion scans:', sum(v > 1 for v in nles.values()))

In [ ]:
# Evaluate every finished run (U-Net++ R4 seeds; AMFU-Net runs too if present) + seed ensembles
import json as _j
UPP = {"arch": "smp:UnetPlusPlus:resnet34"}
probs, rows = {}, []
for r in sorted(os.listdir(RUNS)):
    ck = os.path.join(RUNS, r, 'best.pt')
    if not os.path.exists(ck): continue
    model = g.build_model(UPP if 'upp' in r else {}).to(g.DEVICE)
    model.load_state_dict(torch.load(ck, map_location=g.DEVICE)); model.eval()
    with torch.no_grad():
        for v, tta in (('plain', False), ('tta', True)):
            p = g.predict(model, ext, tta=tta); probs[(r, v)] = p
            d = g.full_metrics(p, ext); rows.append(g.summarise(d, r).assign(variant=v))
            if v == 'plain': d.to_csv(f'ext_{r}_plain.csv', index=False)
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print('done', r)
def ens(name, members, v='tta'):
    ms = [m for m in members if (m, v) in probs]
    if len(ms) < 2: return
    p = np.mean([probs[(m, v)] for m in ms], 0); probs[(name, v)] = p
    d = g.full_metrics(p, ext); rows.append(g.summarise(d, name).assign(variant=v)); d.to_csv(f'ext_{name}.csv', index=False)
runs = sorted({r for r, _ in probs})
for k, pat in {'ENS_upp_ft55': 'upp_ft55_s', 'ENS_upp_dice': 'upp_dice_s', 'ENS_upp_bnd': 'upp_ft55_bnd05_s', 'ENS_upp_all': 'R4_upp_'}.items():
    ens(k, [r for r in runs if pat in r])
board = pd.concat(rows); board = board[board.cohort == 'Overall'].sort_values('dice', ascending=False)
board.to_csv('external_breast_leaderboard.csv', index=False)
print('=== EXTERNAL (BrEaST) LEADERBOARD ===')
print(board[['run', 'variant', 'dice', 'iou', 'hd95', 'asd', 'bf1', 'fail_lt50']].round(2).to_string(index=False))

In [ ]:
%%writefile gallery_core.py
# G17 — segmentation gallery: ground-truth area vs predicted area for one model on the 266-scan test set.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

GT_C, PR_C = "#2ecc71", "#ff4757"


def pick_cases(df, n=30):
    """Evenly spaced ranks over the whole Dice distribution → a representative, unbiased sample
    (best, typical and worst cases in their true proportions)."""
    s = df.sort_values("dice", ascending=False).reset_index()
    idx = np.unique(np.round(np.linspace(0, len(s) - 1, n)).astype(int))
    return s.iloc[idx]


def _shrink(img, *masks, side=520):
    h, w = img.shape
    f = min(1.0, side / max(h, w))
    if f >= 1.0:
        return (img,) + masks
    sz = (int(w * f), int(h * f))
    return (cv2.resize(img, sz, interpolation=cv2.INTER_AREA),) + tuple(
        cv2.resize(m.astype(np.uint8), sz, interpolation=cv2.INTER_NEAREST) > 0 for m in masks)


def overlay(ax, img, mask, colour, title, ref=None):
    ax.imshow(img, cmap="gray", interpolation="lanczos")
    if mask.any():
        rgba = np.zeros(mask.shape + (4,), np.float32)
        rgba[mask] = matplotlib.colors.to_rgba(colour, 0.38)
        ax.imshow(rgba, interpolation="nearest")
        ax.contour(mask, levels=[0.5], colors=colour, linewidths=1.3)
    if ref is not None and ref.any():   # thin dashed ground-truth outline for comparison
        ax.contour(ref, levels=[0.5], colors=GT_C, linewidths=0.8, linestyles="--")
    ax.set_title(title, fontsize=7.5, loc="left")
    ax.axis("off")


def build_gallery(runs_root, run, label, split_test, out_pdf, n=30, variant="plain", per_page=6):
    d = Path(runs_root) / run
    df = pd.read_csv(d / f"test_{variant}.csv").set_index("image_id")
    probs = np.load(d / "test_probs.npz")["tta" if "tta" in variant else "plain"]
    ids = list(pd.read_csv(d / "test_plain.csv").image_id)
    pos = {c: i for i, c in enumerate(ids)}
    native = {r["image_id"]: (np.asarray(r["raw_image"].convert("L")), np.asarray(r["mask"].convert("L")) > 127)
              for r in split_test}
    cases = pick_cases(df, n)
    pdf = PdfPages(out_pdf)
    pngs = []

    # ---- cover page
    fig = plt.figure(figsize=(11.69, 8.27))
    fig.text(0.06, 0.88, "G17 — Lesion segmentation: ground truth vs prediction", fontsize=20, weight="bold")
    fig.text(0.06, 0.83, f"Model: {label}", fontsize=12)
    m = df
    rows = [("Test scans", f"{len(m)}"),
            ("Mean Dice", f"{m.dice.mean()*100:.2f} %"),
            ("Mean IoU", f"{m.jaccard.mean()*100:.2f} %"),
            ("Mean 95% Hausdorff", f"{m.hd95.mean():.2f} px"),
            ("Mean surface distance", f"{m.asd.mean():.2f} px"),
            ("Scans with Dice < 50 %", f"{(m.dice < 0.5).mean()*100:.1f} %")]
    for k, (a, b) in enumerate(rows):
        fig.text(0.08, 0.74 - k * 0.045, a, fontsize=12)
        fig.text(0.36, 0.74 - k * 0.045, b, fontsize=12, weight="bold")
    coh = m.groupby("dataset").dice.agg(["size", "mean"]) if "dataset" in m else None
    if coh is not None:
        fig.text(0.56, 0.74, "By centre", fontsize=12, weight="bold")
        for k, (c, r) in enumerate(coh.iterrows()):
            fig.text(0.56, 0.70 - k * 0.04, f"{c}  (n={int(r['size'])})", fontsize=11)
            fig.text(0.78, 0.70 - k * 0.04, f"{r['mean']*100:.2f} % Dice", fontsize=11)
    fig.text(0.06, 0.38,
             f"The {len(cases)} scans on the following pages were chosen by rank, evenly spaced across the full\n"
             "Dice distribution of the test set (from the best to the worst case), so they show typical behaviour\n"
             "rather than hand-picked successes. Each scan is shown at full frame in its native resolution.\n\n"
             "Left panel: ground-truth lesion area (green).   Right panel: predicted lesion area (red),\n"
             "with the ground-truth outline repeated as a thin dashed green line for comparison.\n"
             "Areas are in pixels of the original image; Dice measures the overlap of the two areas.\n"
             "Threshold 0.5, single forward pass (no post-processing).", fontsize=11, va="top")
    fig.legend(handles=[Patch(color=GT_C, alpha=0.6, label="Ground truth"), Patch(color=PR_C, alpha=0.6, label="Prediction")],
               loc="lower left", bbox_to_anchor=(0.06, 0.08), ncol=2, fontsize=11, frameon=False)
    pdf.savefig(fig); plt.close(fig)

    # ---- case pages: 3 rows × (GT | Pred) × 2 cases per row
    case_list = list(cases.itertuples())
    for p0 in range(0, len(case_list), per_page):
        chunk = case_list[p0:p0 + per_page]
        fig, axes = plt.subplots(3, 4, figsize=(11.69, 8.27))
        for k in range(per_page):
            r, c = divmod(k, 2)
            a_gt, a_pr = axes[r, 2 * c], axes[r, 2 * c + 1]
            if k >= len(chunk):
                a_gt.axis("off"); a_pr.axis("off"); continue
            row = chunk[k]
            cid = row.image_id
            img, gt = native[cid]
            pr = cv2.resize(probs[pos[cid]].astype(np.float32) / 255.0, (gt.shape[1], gt.shape[0]),
                            interpolation=cv2.INTER_LINEAR) >= 0.5
            n_case = p0 + k + 1
            gt_px, pr_px = int(gt.sum()), int(pr.sum())
            img, gt, pr = _shrink(img, gt, pr)
            overlay(a_gt, img, gt, GT_C, f"#{n_case}  {row.dataset} — {cid[:28]}\nGround truth: {gt_px:,} px")
            overlay(a_pr, img, pr, PR_C, f"Dice {row.dice*100:.1f} %  ·  95HD {row.hd95:.1f} px\nPrediction: {pr_px:,} px", ref=gt)
        fig.suptitle(f"{label} — test scans {p0+1}–{p0+len(chunk)} of {len(case_list)} (sorted best → worst)",
                     fontsize=11, x=0.01, ha="left")
        fig.legend(handles=[Patch(color=GT_C, alpha=0.6, label="Ground truth"), Patch(color=PR_C, alpha=0.6, label="Prediction")],
                   loc="lower right", ncol=2, fontsize=9, frameon=False)
        fig.tight_layout(rect=(0, 0.03, 1, 0.95), h_pad=2.6, w_pad=0.6)
        png = str(Path(out_pdf).with_suffix("")) + f"_p{p0 // per_page + 1}.png"
        fig.savefig(png, dpi=130); pngs.append(png)
        pdf.savefig(fig, dpi=150); plt.close(fig)
    pdf.close()
    return pngs


In [ ]:
import gallery_core as G, matplotlib
best = board.iloc[0]; name, v = best.run, best.variant
p = probs[(name, v)]
df = pd.read_csv(f'ext_{name}_plain.csv') if os.path.exists(f'ext_{name}_plain.csv') else pd.read_csv(f'ext_{name}.csv')
d = g.full_metrics(p, ext).set_index('image_id')
# minimal adapter: write a fake run dir so build_gallery can be reused unchanged
os.makedirs('ext_runs/BEST', exist_ok=True)
d.reset_index().to_csv('ext_runs/BEST/test_plain.csv', index=False)
np.savez_compressed('ext_runs/BEST/test_probs.npz', plain=(p * 255).astype(np.uint8), tta=(p * 255).astype(np.uint8))
from PIL import Image as PI
split_like = [{"image_id": k, "raw_image": PI.fromarray(native[k][0]), "mask": PI.fromarray((native[k][1] * 255).astype(np.uint8))} for k in ids]
pngs_out = G.build_gallery('ext_runs', 'BEST', f'{name} ({v}) — EXTERNAL BrEaST', split_like, 'G17_external_BrEaST_gallery.pdf', n=30)
from IPython.display import Image, display
for q in pngs_out: display(Image(q))